In [ ]:
# 1. Install required libraries
!pip install -q transformers datasets librosa evaluate jiwer accelerate peft bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 31.5 MB/s eta 0:00:00


In [7]:
# Upgrade torchao to a compatible version for peft
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 59.7 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [ ]:
# Upgrade transformers to the latest version to ensure compatibility
!pip install --upgrade transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 108.2 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import sys
import torch
import numpy as np
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Set Paths
DRIVE_PATH = "/content/drive/My Drive/NLP - Audio Project"
sys.path.append(DRIVE_PATH)

AUDIO_DIR = os.path.join(DRIVE_PATH, "cut_clips9")
TRANS_DIR = os.path.join(DRIVE_PATH, "cut_clips9_transcriptions")
OUTPUT_DIR = os.path.join(DRIVE_PATH, "whisper-small-egyptian-lora-final")

from data_loader import EgyptDatasetBuilder

In [ ]:
import os

# 3. Load Dataset
MODEL_NAME = "openai/whisper-small"
processor = WhisperProcessor.from_pretrained(MODEL_NAME, language="arabic", task="transcribe")

DATASET_CACHE_DIR = os.path.join(DRIVE_PATH, "dataset_cache")

if os.path.exists(DATASET_CACHE_DIR):
    print(f"Loading dataset from cache: {DATASET_CACHE_DIR}")
    from datasets import load_from_disk
    dataset = load_from_disk(DATASET_CACHE_DIR)
else:
    print("Building dataset from scratch and saving to cache...")
    builder = EgyptDatasetBuilder(AUDIO_DIR, TRANS_DIR)
    dataset = builder.build()
    dataset = dataset.train_test_split(test_size=100, seed=42)
    dataset.save_to_disk(DATASET_CACHE_DIR)
    print(f"Dataset saved to cache: {DATASET_CACHE_DIR}")

print(f"Dataset loaded. Train samples: {len(dataset['train'])}")
print(f"Test samples: {len(dataset['test'])}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading dataset from cache: /content/drive/My Drive/NLP - Audio Project/dataset_cache
Dataset loaded. Train samples: 7401
Test samples: 100


In [ ]:
!pip install -U bitsandbytes>=0.46.1

In [ ]:
# 4. Model Initialization (4-bit QLoRA for Colab T4)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto'
)

model = prepare_model_for_kbit_training(model)

config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=['q_proj', 'v_proj'],
    lora_dropout=0.05,
    bias='none'
)

model = get_peft_model(model, config)
model.generation_config.language = "arabic"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

In [ ]:
import evaluate
from dataclasses import dataclass
from typing import Any, Dict, List
import librosa

metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)
    wer = 100 * metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        input_features = []
        label_features = []
        for feature in features:
            # The 'audio' column contains the path, which we load here
            audio_array, _ = librosa.load(feature['audio'], sr=16000)
            f = self.processor.feature_extractor(audio_array, sampling_rate=16000).input_features[0]
            input_features.append({'input_features': f})
            l = self.processor.tokenizer(feature['sentence']).input_ids
            label_features.append({'input_ids': l})

        batch = self.processor.feature_extractor.pad(input_features, return_tensors='pt')
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors='pt')
        labels = labels_batch['input_ids'].masked_fill(labels_batch.attention_mask.ne(1), -100)
        batch['labels'] = labels
        return batch

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=6,
    per_device_train_batch_size=32,
    gradient_accumulation_steps=2,
    learning_rate=1e-4,
    warmup_steps=50,
    gradient_checkpointing=True,
    fp16=True,
    eval_strategy='steps',
    per_device_eval_batch_size=4,
    predict_with_generate=True,
    generation_max_length=180,
    save_steps=100,
    eval_steps=100,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model='wer',
    greater_is_better=False,
    remove_unused_columns=False,
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test'],
    data_collator=DataCollatorSpeechSeq2SeqWithPadding(processor=processor),
    processing_class=processor.feature_extractor,
    compute_metrics=compute_metrics,
)

In [ ]:
print("Starting Training on Colab... (Checkpoints will be in your Drive)")
trainer.train(resume_from_checkpoint=True)

Starting Training on Colab... (Checkpoints will be in your Drive)


Step,Training Loss,Validation Loss


TrainOutput(global_step=696, training_loss=0.08122965830495987, metrics={'train_runtime': 6390.862, 'train_samples_per_second': 6.948, 'train_steps_per_second': 0.109, 'total_flos': 1.304121978667008e+19, 'train_loss': 0.08122965830495987, 'epoch': 6.0})

In [ ]:
# Save final model
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print(f"Training finished! Model saved to {OUTPUT_DIR}")

Training finished! Model saved to /content/drive/My Drive/NLP - Audio Project/whisper-small-egyptian-lora-final


# Pruning

In [ ]:
import torch.nn.utils.prune as prune

def apply_pruning_to_lora(model, amount=0.3):
    """
    Prunes a percentage of the LoRA adapter weights.
    amount: float between 0.0 and 1.0 (e.g., 0.3 = 30% pruning)
    """
    print(f"Applying {amount*100}% Magnitude Pruning to LoRA layers...")

    # We target the LoRA 'A' and 'B' matrices in the attention layers
    for name, module in model.named_modules():
        if "lora_A" in name or "lora_B" in name:
            if isinstance(module, torch.nn.Linear):
                prune.l1_unstructured(module, name="weight", amount=amount)
                # Make the pruning permanent
                prune.remove(module, "weight")

    print("Pruning complete.")
    return model

In [ ]:
import os
import sys
import time
import torch
import librosa
import evaluate
import pandas as pd
import torch.nn.utils.prune as prune
from IPython.display import display, HTML
from transformers import logging, WhisperProcessor, WhisperForConditionalGeneration
from peft import PeftModel
from datasets import load_from_disk # Import load_from_disk

# --- 1. SETUP & PATHS ---
# Add parent directory to path to import your custom data_loader
sys.path.append(os.path.abspath('..'))
from data_loader import EgyptDatasetBuilder

# Assuming DRIVE_PATH is defined earlier or redefine it here for self-containment
DRIVE_PATH = "/content/drive/My Drive/NLP - Audio Project"

MODEL_FOLDER = os.path.join(DRIVE_PATH, "whisper-small-egyptian-lora-final")
BASE_MODEL_NAME = "openai/whisper-small"
AUDIO_DIR = os.path.join(DRIVE_PATH, "cut_clips9")
TRANS_DIR = os.path.join(DRIVE_PATH, "cut_clips9_transcriptions")
DATASET_CACHE_DIR = os.path.join(DRIVE_PATH, "dataset_cache") # Define DATASET_CACHE_DIR

# Detect device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Initializing on {device.upper()}...")

# --- 2. INITIALIZE DATASET & METRIC ---
logging.set_verbosity_error()

# Initialize Metric (Word Error Rate)
metric = evaluate.load("wer")

# Build Dataset (using cache logic)
if os.path.exists(DATASET_CACHE_DIR):
    print(f"Loading dataset from cache: {DATASET_CACHE_DIR}")
    dataset = load_from_disk(DATASET_CACHE_DIR)
else:
    print("Building dataset from scratch and saving to cache...")
    builder = EgyptDatasetBuilder(AUDIO_DIR, TRANS_DIR)
    full_dataset = builder.build()
    dataset = full_dataset.train_test_split(test_size=100, seed=42)
    dataset.save_to_disk(DATASET_CACHE_DIR)
    print(f"Dataset saved to cache: {DATASET_CACHE_DIR}")

test_subset = dataset["test"].select(range(min(20, len(dataset["test"]))))
print(f"✅ Dataset prepared. Using {len(test_subset)} samples for benchmarking.")

# --- 3. LOAD MODEL & PROCESSOR ---
print(f"📦 Loading model from {MODEL_FOLDER}...")
processor = WhisperProcessor.from_pretrained(MODEL_FOLDER)

# Load base model in float32 for CPU/GPU compatibility
model = WhisperForConditionalGeneration.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=torch.float32,
    low_cpu_mem_usage=True
)
# Load LoRA Adapters
model = PeftModel.from_pretrained(model, MODEL_FOLDER)
model.to(device)
model.eval()

# Configure Generation
model.generation_config.max_length = None
model.generation_config.language = "arabic"
model.generation_config.task = "transcribe"

# --- 4. HELPER FUNCTIONS ---

def count_nonzero_params(model):
    nonzero = 0
    total = 0
    for param in model.parameters():
        nonzero += torch.count_nonzero(param).item()
        total += param.nelement()
    return nonzero, total

def get_stats():
    mem_usage = torch.cuda.memory_allocated() / 1024**2 if torch.cuda.is_available() else 0
    nonzero, _ = count_nonzero_params(model)
    theoretical_mb = (nonzero * 4) / 1024**2
    return mem_usage, theoretical_mb

def get_lora_state_dict(model):
    return {k: v.cpu().clone() for k, v in model.named_parameters() if "lora" in k}

def run_mini_benchmark(active_model, desc):
    predictions = []
    references = []

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    start_mem, theoretical_size = get_stats()
    start_time = time.time()

    for item in test_subset:
        audio_path = item["audio"]
        audio, _ = librosa.load(audio_path, sr=16000)
        input_features = processor(audio, sampling_rate=16000, return_tensors="pt").input_features.to(device)

        with torch.no_grad():
            generated_ids = active_model.generate(input_features, max_new_tokens=128)

        pred = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
        predictions.append(pred)
        references.append(item["sentence"])

    elapsed = time.time() - start_time
    wer_score = 100 * metric.compute(predictions=predictions, references=references)

    return {
        "Configuration": desc,
        "WER (%)": f"{wer_score:.2f}%",
        "Total Time (s)": f"{elapsed:.2f}s",
        "Avg Latency (s)": f"{elapsed/len(test_subset):.2f}s",
        "Memory (MB)": f"{start_mem:.0f} MB" if torch.cuda.is_available() else "N/A (CPU)",
        "Sparse Size (MB)": f"{theoretical_size:.1f} MB"
    }

# --- 5. MASTER EXECUTION LOOP ---
results = []
original_lora_state = get_lora_state_dict(model)

def restore_lora():
    model.load_state_dict(original_lora_state, strict=False)

# TEST 1: Baseline
print("Testing Baseline...")
results.append(run_mini_benchmark(model, "Baseline (Full Fine-tuned)"))

# TEST 2: Magnitude Pruning (30%)
# If you don't have apply_pruning_to_lora defined elsewhere, you can paste it here
def apply_pruning_to_lora(model, amount=0.3):
    for name, module in model.named_modules():
        if "lora_A" in name or "lora_B" in name:
            if isinstance(module, torch.nn.Linear):
                prune.l1_unstructured(module, name="weight", amount=amount)
                prune.remove(module, "weight")
    return model

print("Applying Pruning...")
apply_pruning_to_lora(model, amount=0.3)
results.append(run_mini_benchmark(model, "Magnitude Pruned (30%)"))
restore_lora()

# TEST 3: Layer Pruning
print("Testing Layer Pruning...")
original_decoder_layers = model.base_model.model.model.decoder.layers
model.base_model.model.model.decoder.layers = torch.nn.ModuleList(list(original_decoder_layers)[:-2])
results.append(run_mini_benchmark(model, "Layer Pruned (-2 Decoder Layers)"))
model.base_model.model.model.decoder.layers = original_decoder_layers # Restore

# 📊 FINAL COMPARISON TABLE
df_master = pd.DataFrame(results)
display(HTML("<h2 style='color: #2c3e50;'>🚀 Master Efficiency Benchmark Results</h2>"))
display(df_master)

🚀 Initializing on CUDA...
Loading dataset from cache: /content/drive/My Drive/NLP - Audio Project/dataset_cache
✅ Dataset prepared. Using 20 samples for benchmarking.
📦 Loading model from /content/drive/My Drive/NLP - Audio Project/whisper-small-egyptian-lora-final...


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Testing Baseline...
Applying Pruning...
Testing Layer Pruning...


,Configuration,WER (%),Total Time (s),Avg Latency (s),Memory (MB),Sparse Size (MB)
0,Baseline (Full Fine-tuned),33.65%,10.78s,0.54s,1589 MB,935.6 MB
1,Magnitude Pruned (30%),34.62%,10.89s,0.54s,1589 MB,931.6 MB
2,Layer Pruned (-2 Decoder Layers),100.00%,56.01s,2.80s,1589 MB,862.0 MB


# Comparison

In [ ]:
import time
import os
import torch
import librosa
import evaluate
import pandas as pd
from IPython.display import display, HTML
from transformers import pipeline, WhisperForConditionalGeneration
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    BitsAndBytesConfig
)

# Detect device - ensure it's defined within this cell's scope
device = "cuda" if torch.cuda.is_available() else "cpu"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)


# 1. 📋 CONFIGURE TEST DATA
COMPARISON_FOLDER = "/content/audio_comparison"
TRUE_LABELS = {
    "sample1.wav": "منور يا باشا عامل ايه",
    "sample2.wav": "دلوقتي هنبدا الدرس التاني",
    "sample3.wav": "يا ريت تفكك من الحوارات دي علشان مش جايبة همها",
    "sample4.wav": "هو كل شوية يطنشني ويقولي هبقى أكلمك بعدين",
    "sample5.wav": "إيه يا عم القفش ده؟ خليك فريش شوية",
    "sample6.wav": "الموضوع بييجي خبط لزق كده من غير مقدمات",
    "sample7.wav": "على وضعك يا يعم ومتشيلش من حد",
    "sample8.wav": "بقولك إيه سيبك من اللت والعجن الكتير ده",
    "sample9.wav": "عاش يا وحش التطور ده بييجي بالممارسة",
    "sample10.wav": "متكبرهاش في دماغك بقى وصفي نيتك",
}

wer_metric = evaluate.load("wer")

def get_model_size_mb(model):
    param_size = sum(p.nelement() * p.element_size() for p in model.parameters())
    return param_size / 1024**2

def benchmark_models():
    models_to_test = [
        {"name": "Original Small", "path": "openai/whisper-small", "type": "base"},
        {"name": "Fine-tuned Student", "type": "lora"},
        {"name": "Pruned Student (30%)", "type": "lora_pruned", "amount": 0.3},
        {"name": "Teacher Large-v3", "path": "openai/whisper-large-v3", "type": "pipe"}
    ]

    overall_results = []
    # This will store: { "sample1.wav": {"Reference": "...", "Original Small": "...", ...} }
    transcription_comparison = {k: {"Reference": v} for k, v in TRUE_LABELS.items()}

    original_lora_state = {k: v.cpu().clone() for k, v in model.named_parameters() if "lora" in k}

    for m_cfg in models_to_test:
        print(f"🚀 Benchmarking: {m_cfg['name']}...")
        current_model, pipe = None, None

        # --- LOAD ---
        if m_cfg['type'] == "base":
            current_model = WhisperForConditionalGeneration.from_pretrained(
                m_cfg['path'], quantization_config=bnb_config, device_map="auto"
            )
            current_model.to(device)
        elif m_cfg['type'] == "lora":
            current_model = model
            current_model.to(device) # Ensure the model is on the correct device
        elif m_cfg['type'] == "lora_pruned":
            current_model = model
            apply_pruning_to_lora(current_model, amount=m_cfg['amount'])
            current_model.to(device) # Ensure the model is on the correct device
        elif m_cfg['type'] == "pipe":
            pipe = pipeline("automatic-speech-recognition", model=m_cfg['path'],
                            model_kwargs={"quantization_config": bnb_config, "device_map": "auto"})

        # --- STATS ---
        torch.cuda.empty_cache()
        vram = torch.cuda.memory_allocated() / 1024**2
        model_size = get_model_size_mb(current_model) if current_model else 1550

        # --- INFERENCE ---
        preds, refs = [], []
        start_time = time.time()

        for filename, truth in TRUE_LABELS.items():
            path = os.path.join(COMPARISON_FOLDER, filename)
            if not os.path.exists(path): continue

            audio, _ = librosa.load(path, sr=16000)
            if pipe:
                pred = pipe(audio)["text"]
            else:
                input_features = processor(audio, sampling_rate=16000, return_tensors="pt").input_features.to(device)
                with torch.no_grad():
                    gen_ids = current_model.generate(input_features, max_new_tokens=128)
                pred = processor.batch_decode(gen_ids, skip_special_tokens=True)[0]

            preds.append(pred)
            refs.append(truth)
            # Store for the transcription table
            transcription_comparison[filename][m_cfg['name']] = pred

        # --- METRICS ---
        if preds:
            elapsed = time.time() - start_time
            wer = 100 * wer_metric.compute(predictions=preds, references=refs)
            overall_results.append({
                "Model": m_cfg['name'], "WER (%)": f"{wer:.2f}%",
                "Avg Time (s)": f"{elapsed/len(preds):.2f}s",
                "VRAM (MB)": f"{vram:.0f} MB", "Size (MB)": f"{model_size:.0f} MB"
            })

        # --- CLEANUP ---
        if m_cfg['type'] == "lora_pruned":
            model.load_state_dict(original_lora_state, strict=False)
        if m_cfg['type'] in ["base", "pipe"]:
            del current_model, pipe
            torch.cuda.empty_cache()

    # --- RENDER STYLING ---
    styles = """
    <style>
        .benchmark-container { background-color: #1e1e2e; color: #cdd6f4; padding: 20px; border-radius: 15px; font-family: 'Segoe UI', sans-serif; }
        .premium-table { border-collapse: collapse; width: 100%; background-color: #313244; border-radius: 10px; overflow: hidden; margin-bottom: 20px; }
        .premium-table th { background-color: #45475a; color: #89b4fa; padding: 15px; text-align: left; text-transform: uppercase; font-size: 0.85em; }
        .premium-table td { border-bottom: 1px solid #45475a; padding: 12px; color: #cdd6f4; }
        .premium-table tr:hover { background-color: #45475a; transition: 0.3s; }
        .ref-col { color: #a6e3a1 !important; font-weight: bold; }
        h2 { color: #f5c2e7; margin-bottom: 10px; }
    </style>
    """

    # 📊 1. METRICS SUMMARY
    df_metrics = pd.DataFrame(overall_results)

    # 📝 2. TRANSCRIPTION DETAILS
    # Flatten the nested dict for pandas
    detailed_data = []
    for fname, data in transcription_comparison.items():
        row = {"File": fname}
        row.update(data)
        detailed_data.append(row)
    df_details = pd.DataFrame(detailed_data)

    display(HTML(styles + f"""
    <div class='benchmark-container'>
        <h2>🚀 Model Performance Metrics</h2>
        {df_metrics.to_html(index=False, classes='premium-table')}

        <h2>📝 Detailed Transcriptions Comparison</h2>
        {df_details.to_html(index=False, classes='premium-table')}
    </div>
    """))

In [ ]:
benchmark_models()

🚀 Benchmarking: Original Small...


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

🚀 Benchmarking: Fine-tuned Student...
🚀 Benchmarking: Pruned Student (30%)...
🚀 Benchmarking: Teacher Large-v3...


Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

In [8]:
import os
import torch
import torch.nn.utils.prune as prune
from transformers import WhisperProcessor, WhisperForConditionalGeneration, AutoFeatureExtractor, AutoTokenizer
from peft import PeftModel

DRIVE_PATH = "/content/drive/My Drive/NLP - Audio Project"

def apply_pruning_to_lora(model, amount=0.3):
    """
    Prunes a percentage of the LoRA adapter weights.
    amount: float between 0.0 and 1.0 (e.g., 0.3 = 30% pruning)
    """
    print(f"Applying {amount*100}% Magnitude Pruning to LoRA layers...")

    # We target the LoRA 'A' and 'B' matrices in the attention layers
    for name, module in model.named_modules():
        if "lora_A" in name or "lora_B" in name:
            if isinstance(module, torch.nn.Linear):
                prune.l1_unstructured(module, name="weight", amount=amount)
                # Make the pruning permanent
                prune.remove(module, "weight")

    print("Pruning complete.")
    return model

# Define paths and names
MODEL_FOLDER = os.path.join(DRIVE_PATH, "whisper-small-egyptian-lora-final")
BASE_MODEL_NAME = "openai/whisper-small"
PRUNED_MODEL_OUTPUT_DIR = os.path.join(DRIVE_PATH, "whisper-small-egyptian-lora-pruned-30")

# Ensure output directory exists
os.makedirs(PRUNED_MODEL_OUTPUT_DIR, exist_ok=True)

# Detect device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Initializing on {device.upper()}...")

# Load processor components from the base model
print(f"📦 Loading feature extractor from {BASE_MODEL_NAME}...")
feature_extractor = AutoFeatureExtractor.from_pretrained(BASE_MODEL_NAME)

print(f"📦 Loading tokenizer from {BASE_MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, language="arabic", task="transcribe")

processor = WhisperProcessor(feature_extractor=feature_extractor, tokenizer=tokenizer)

# Load base model and LoRA adapters
print(f"📦 Loading model from {MODEL_FOLDER}...")
model = WhisperForConditionalGeneration.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=torch.float32,
    low_cpu_mem_usage=True
)
model = PeftModel.from_pretrained(model, MODEL_FOLDER, local_files_only=True)
model.to(device)
model.eval()

# Configure Generation
model.generation_config.max_length = None
model.generation_config.language = "arabic"
model.generation_config.task = "transcribe"

print("Applying 30% pruning to the fine-tuned model...")
pruned_model = apply_pruning_to_lora(model, amount=0.3)

print(f"Saving pruned model to {PRUNED_MODEL_OUTPUT_DIR}...")
pruned_model.save_pretrained(PRUNED_MODEL_OUTPUT_DIR)
processor.save_pretrained(PRUNED_MODEL_OUTPUT_DIR)

print("Pruned model and processor saved successfully!")

🚀 Initializing on CUDA...
📦 Loading feature extractor from openai/whisper-small...
📦 Loading tokenizer from openai/whisper-small...
📦 Loading model from /content/drive/My Drive/NLP - Audio Project/whisper-small-egyptian-lora-final...


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Applying 30% pruning to the fine-tuned model...
Applying 30.0% Magnitude Pruning to LoRA layers...
Pruning complete.
Saving pruned model to /content/drive/My Drive/NLP - Audio Project/whisper-small-egyptian-lora-pruned-30...
Pruned model and processor saved successfully!
